preprocessing

In [1]:
import re
import unicodedata

INPUT_FILE = "te.txt"
OUTPUT_FILE = "te_preprocessed.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    count = 0

    for line in fin:

        # Remove leading/trailing whitespace
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Unicode normalization
        line = unicodedata.normalize("NFC", line)

        # Replace multiple spaces/tabs/newlines with one space
        line = re.sub(r"\s+", " ", line)

        # Remove control characters
        line = re.sub(r"[\x00-\x1F\x7F]", "", line)

        fout.write(line + "\n")

        count += 1

        if count % 100000 == 0:
            print(f"Processed {count:,} lines")

print("Preprocessing completed.")

Processed 100,000 lines
Processed 200,000 lines
Processed 300,000 lines
Processed 400,000 lines
Processed 500,000 lines
Processed 600,000 lines
Processed 700,000 lines
Processed 800,000 lines
Processed 900,000 lines
Processed 1,000,000 lines
Processed 1,100,000 lines
Processed 1,200,000 lines
Processed 1,300,000 lines
Processed 1,400,000 lines
Processed 1,500,000 lines
Processed 1,600,000 lines
Processed 1,700,000 lines
Processed 1,800,000 lines
Processed 1,900,000 lines
Processed 2,000,000 lines
Processed 2,100,000 lines
Processed 2,200,000 lines
Processed 2,300,000 lines
Processed 2,400,000 lines
Processed 2,500,000 lines
Processed 2,600,000 lines
Processed 2,700,000 lines
Processed 2,800,000 lines
Processed 2,900,000 lines
Processed 3,000,000 lines
Processed 3,100,000 lines
Processed 3,200,000 lines
Processed 3,300,000 lines
Processed 3,400,000 lines
Processed 3,500,000 lines
Processed 3,600,000 lines
Processed 3,700,000 lines
Processed 3,800,000 lines
Processed 3,900,000 lines
Proc

SENTENCE TOKENIZATION

In [11]:
import re
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pattern = re.compile(r'(?<=[.!?])\s+')

input_file = "te_preprocessed.txt"
output_file = "all_sentences.parquet"

BATCH_SIZE = 100000   # Number of sentences per batch

writer = None
batch = []

total_sentences = 0
total_length = 0

TOKEN_PATTERN = re.compile(r"""
    https?://[^\s]+|                       # URLs
    \b[\w.+-]+@[\w.-]+\.\w+\b|             # Email IDs
    \d{1,2}[/-]\d{1,2}[/-]\d{2,4}|         # Dates
    \d+\.\d+|                             # Decimal numbers
    \d+|                                  # Integers
    [\u0C00-\u0C7F]+|                     # Telugu words
    [A-Za-z]+|                            # English words
    \.{3}|                                # Ellipsis (...)
    [.,!?;:()"'\-]                        # Punctuation
""", re.VERBOSE)

with open(input_file, "r", encoding="utf-8") as fin:

    for line in fin:
        line = line.strip()

        if not line:
            continue

        sentences = [s.strip() for s in pattern.split(line) if s.strip()]

        for sentence in sentences:
            batch.append(sentence)
            total_sentences += 1
            total_words=len(TOKEN_PATTERN.findall(sentence))
            total_length+=total_words

            if len(batch) >= BATCH_SIZE:
                df = pd.DataFrame({"text": batch})

                table = pa.Table.from_pandas(df)

                if writer is None:
                    writer = pq.ParquetWriter(output_file, table.schema)

                writer.write_table(table)

                batch.clear()

if batch:
    df = pd.DataFrame({"text": batch})
    table = pa.Table.from_pandas(df)

    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)

    writer.write_table(table)

if writer is not None:
    writer.close()

print("Total sentences:", total_sentences)
print("Average sentence length:", total_length / total_sentences)
print("Saved to:", output_file)

Total sentences: 77685001
Average sentence length: 11.61605206132391
Saved to: all_sentences.parquet


WORD TOKENIZATION

In [6]:
import re
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


# Input / Output

INPUT_FILE = "all_sentences.parquet"
OUTPUT_FILE = "all_words.parquet"

BATCH_SIZE = 100000


# Word Tokenizer Regex(pattern)

TOKEN_PATTERN = re.compile(r"""
    https?://[^\s]+|                       # URLs
    \b[\w.+-]+@[\w.-]+\.\w+\b|             # Email IDs
    \d{1,2}[/-]\d{1,2}[/-]\d{2,4}|         # Dates
    \d+\.\d+|                             # Decimal numbers
    \d+|                                  # Integers
    [\u0C00-\u0C7F]+|                     # Telugu words
    [A-Za-z]+|                            # English words
    \.{3}|                                # Ellipsis (...)
    [.,!?;:()"'\-]                        # Punctuation
""", re.VERBOSE)


# Statistics
total_words = 0
total_characters = 0
unique_tokens = set()

# Writer
writer = None
batch = []


parquet = pq.ParquetFile(INPUT_FILE)

for rg in range(parquet.num_row_groups):

    df = parquet.read_row_group(rg).to_pandas()

    for sentence in df["text"]:

        tokens = TOKEN_PATTERN.findall(sentence)

        total_words += len(tokens)

        for token in tokens:

            batch.append(token)

            total_characters += len(token)

            unique_tokens.add(token)

            if len(batch) >= BATCH_SIZE:

                temp = pd.DataFrame({"text": batch})

                table = pa.Table.from_pandas(temp)

                if writer is None:
                    writer = pq.ParquetWriter(
                        OUTPUT_FILE,
                        table.schema
                    )

                writer.write_table(table)

                batch.clear()

# Write remaining tokens
if batch:

    temp = pd.DataFrame({"text": batch})

    table = pa.Table.from_pandas(temp)

    if writer is None:
        writer = pq.ParquetWriter(
            OUTPUT_FILE,
            table.schema
        )

    writer.write_table(table)

if writer:
    writer.close()

print("Total words:", total_words)
print("Total characters:", total_characters)
print("Average word length:", total_characters / total_words)
print("Unique tokens:", len(unique_tokens))
print("TTR:", len(unique_tokens) / total_words)
print("Saved to:", OUTPUT_FILE)

Total words: 902393016
Total characters: 5085331598
Average word length: 5.6353844808568425
Unique tokens: 7916018
TTR: 0.008772250958999
Saved to: all_words.parquet
